# Linha do tempo dos conceitos no ensino de Lacan

Frequência relativa (ocorrências por 10 mil palavras) de conceitos ao longo dos seminários,
sessão a sessão (Staferla, francês). Ver `docs/adr/0002-medicao-de-conceitos.md` para os limites
da medida: a curva mostra **onde ler**, não o que o conceito significa.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import psycopg

conn = psycopg.connect(os.environ["DATABASE_URL"])

def q(sql, **params):
    return pd.read_sql(sql, conn, params=params)

## Por seminário (Lacan, todas as fontes)

In [ ]:
conceitos = ["gozo", "objeto-a", "nome-do-pai", "significante", "sinthoma", "no-borromeano", "real"]

por_seminario = q('''
    select concept, seminar_number, min(period_date) as inicio,
           10000.0 * sum(mentions) / nullif(sum(words), 0) as por_10k
    from mv_concept_timeline
    where line = 'lacan' and seminar_number is not null and concept = any(%(c)s)
    group by 1, 2 order by 3''', c=conceitos)

tabela = por_seminario.pivot_table(index=["inicio", "seminar_number"], columns="concept", values="por_10k")
tabela.round(1)

In [ ]:
ax = tabela.reset_index(level=0, drop=True).plot(figsize=(14, 5), marker="o")
ax.set_ylabel("ocorrências por 10 mil palavras")
ax.set_xlabel("seminário")
ax.set_title("Conceitos ao longo dos seminários de Lacan")
plt.xticks(range(len(tabela)), tabela.index.get_level_values(1))
plt.show()

## Sessão a sessão (Staferla) — um conceito

In [ ]:
conceito = "gozo"
sessoes = q('''
    select period_date, seminar_number, 10000.0 * sum(mentions) / nullif(sum(words), 0) as por_10k
    from mv_concept_timeline
    where concept = %(c)s and source_id = 'staferla' and period_precision = 'day'
    group by 1, 2 having sum(words) > 1500 order by 1''', c=conceito)

sessoes["media_movel"] = sessoes["por_10k"].rolling(8, min_periods=3).mean()
ax = sessoes.plot(x="period_date", y=["por_10k", "media_movel"], figsize=(14, 4), alpha=.8)
ax.set_title(f"'{conceito}' por sessão (Staferla)")
plt.show()

## Rede de conceitos (PMI)

In [ ]:
rede = q('''
    with tot as (select count(distinct chunk_id)::float n from concept_mention),
         f as (select concept_id, count(*)::float n from concept_mention group by 1)
    select a.slug as c1, b.slug as c2, co.n_chunks,
           ln(co.n_chunks * tot.n / (fa.n * fb.n)) as pmi
    from mv_concept_cooccurrence co
    join concept a on a.id = co.c1 join concept b on b.id = co.c2
    join f fa on fa.concept_id = co.c1 join f fb on fb.concept_id = co.c2, tot
    where co.n_chunks >= 20 order by pmi desc limit 40''')
rede